# Prompt-sensitivity check — do the internals depend on the few-shot block?

**The question.** In `paper_figures.ipynb` every figure inherits one caveat: `correct` is
scored on the **15-shot** prompt, while every internal quantity — logit lens, `gold_rank`,
`gold_prob`, entropy, margin, hidden states, neuron dump — is measured on the **zero-shot**
cloze. Among facts the 15-shot decode scores correct, only 40.8% have the gold token as the
zero-shot argmax, so the event and the measurement live on different inputs.

That is a deliberate design choice (`src/recall.py`: *"gold_prob is always measured on the
zero-shot cloze regardless, so it stays comparable across shot counts"*), and there are real
arguments for it — the answer position is unconditioned by whichever demonstrations were
sampled, and under `drop-same-prop` the 15-shot prefix is **relation-dependent**, which would
inject relation identity into the very representation whose relation-dependence Fig. A3 is
trying to bound.

But the caveat is currently **unquantified**, and it does not need to be. The demonstration
block is deterministic (`src/fewshot.py`: *"Selection rule: Deterministic, no RNG"*) — 15
distinct prefixes, one per relation — so the same lens can be run on the 15-shot prompt and the
two compared directly. The answer position is just the last token of whichever prompt is used;
there is no localisation problem.

**What this notebook does.** At a single checkpoint, run the logit lens on both prompts for
every scoreable fact and ask two things:

1. **Per fact** — how well do `gold_rank`, `gold_prob`, entropy and `is_top1` agree across the
   two prompts?
2. **Per figure** — does Figure 1's survive-vs-forget depth profile keep its *shape* when the
   lens is run on the prompt that actually produced the labels?

Question 2 is the one that matters for the paper. If the gap curves agree, the caveat becomes a
measured footnote. If they do not, Figure 1 needs to be recomputed or explicitly scoped.

One checkpoint is 1/13 the cost of the full sweep. Nothing here writes to the paper artifacts.

In [ ]:
# ── setup ─────────────────────────────────────────────────────────────────────────────
import glob, re, sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

sys.path.insert(0, ".")
from src import data as D
from src import config
from src.fewshot import FewShot

# ---- parameters ----------------------------------------------------------------------
# STEP must have a SUCCESSOR checkpoint, or there are no survive-vs-forget transitions to
# profile.  600000 is the last such step and the closest to the final model.
STEP = 600_000
NEXT_STEP = 658_032
TRAJ_DIR = "data/artifacts/recall/rotemab_15shot_top10_internal"
OUT = Path("data/artifacts/prompt_sensitivity")
LIMIT = None          # set to e.g. 40 for a smoke test before committing to the full run
NUM_SHOTS = 15        # must match the run that produced TRAJ_DIR

# style, shared with paper_figures.ipynb
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
BLUE, ORANGE, PURPLE, NEUTRAL = "#2a78d6", "#eb6834", "#7b5ea7", "#f0efec"
plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "text.color": INK, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.edgecolor": GRID, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False,
})

def clean(ax, axis="y"):
    ax.set_axisbelow(True); ax.grid(axis=axis, alpha=.4); ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)

OUT.mkdir(parents=True, exist_ok=True)
print(f"checkpoint {STEP} -> {NEXT_STEP}")

In [ ]:
# ── the fact set: exactly the facts the paper panel uses ──────────────────────────────
# Taking ids straight from the artifact guarantees the same set as paper_figures.ipynb, without
# re-deriving the --top-shared-chunks-pct / drop_subjects filters here.
def _parq(step):
    hits = glob.glob(f"{TRAJ_DIR}/recall_step{step}_*.parquet")
    assert len(hits) == 1, f"{len(hits)} parquets for step {step}"
    return pd.read_parquet(hits[0])

cur_p, nxt_p = _parq(STEP), _parq(NEXT_STEP)
cur_p = cur_p[~cur_p["excluded"]].set_index("id")
nxt_p = nxt_p[~nxt_p["excluded"]].set_index("id")
ids = cur_p.index.intersection(nxt_p.index)
cur_p, nxt_p = cur_p.loc[ids], nxt_p.loc[ids]

full = D.load_popqa_df()
# The demonstration pool is chosen from the FULL dataset, exactly as eval_checkpoints.py does,
# so prefix_for(row) reproduces the block that was actually used at eval time.
fewshot = FewShot.build(full, num_shots=NUM_SHOTS, source="ours",
                        mode="drop-same-prop", verbose=False)
rows = full.set_index("id").loc[ids].reset_index()
if LIMIT:
    rows = rows.head(LIMIT)
    ids = pd.Index(rows["id"])
    cur_p, nxt_p = cur_p.loc[ids], nxt_p.loc[ids]

correct_now = cur_p["correct"].to_numpy().astype(bool)
forget_next = correct_now & ~nxt_p["correct"].to_numpy().astype(bool)
print(f"{len(rows)} facts | correct at {STEP}: {correct_now.sum()} | forgotten by "
      f"{NEXT_STEP}: {forget_next.sum()} ({forget_next.sum() / max(correct_now.sum(), 1):.1%})")

# Verify the reconstructed 15-shot prompt matches the one the artifact recorded.  If this fails,
# the few-shot configuration here differs from the run and nothing below is comparable.
_chk = rows.head(200).apply(lambda r: fewshot.prompt(r) == cur_p.loc[r["id"], "prompt"], axis=1)
assert _chk.all(), f"prompt mismatch on {(~_chk).sum()}/{len(_chk)} facts -- check NUM_SHOTS/mode"
print(f"prompt reconstruction verified on {len(_chk)} facts")

In [ ]:
# ── the lens, run on an arbitrary prompt ──────────────────────────────────────────────
# A prompt-agnostic version of src.features.internal.extract_fact.  Two differences from the
# original beyond the prompt argument:
#   * it takes the prompt as a parameter instead of hardcoding the zero-shot cloze;
#   * it does NOT re-apply the final norm to hidden_states[-1], which already arrives post-norm.
#     (internal.py now carries the same fix; the stored parquets predate it, which is why
#     paper_figures.ipynb repairs boundary 18 from the `gold_rank` column at load time.)
model = D.load_checkpoint(STEP)
tokenizer = D.load_tokenizer()
norm = getattr(getattr(model, "model", model), "norm", None)
head = getattr(model, "lm_head", None) or model.get_output_embeddings()
print(model.config.num_hidden_layers, "layers |", next(model.parameters()).dtype,
      "|", next(model.parameters()).device)


@torch.no_grad()
def lens(prompt: str, obj: str):
    """Answer-position logit lens + final-layer scalars for one prompt."""
    gold = D.answer_token_ids(obj, tokenizer)
    if not gold:
        return None
    ids_t = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    out = model(ids_t, output_hidden_states=True)
    hs, pos, n_hs = out.hidden_states, ids_t.shape[1] - 1, len(out.hidden_states)
    g0 = gold[0]

    final = out.logits[0, pos].float()
    logp = torch.log_softmax(final, -1)
    p = logp.exp()
    top1 = int(final.argmax())
    tmp = final.clone()
    tmp[g0] = -float("inf")
    span = torch.log_softmax(out.logits[0, pos:pos + len(gold)].float(), -1)
    tok_lp = span[torch.arange(len(gold)), torch.tensor(gold, device=span.device)]

    rank_by_layer = []
    for L, h in enumerate(hs):
        v = h[0, pos]
        v = norm(v) if (norm is not None and L < n_hs - 1) else v   # no double norm at the top
        ll = head(v).float()
        rank_by_layer.append(int((ll > ll[g0]).sum()))
    return {
        "gold_rank": int((final > final[g0]).sum()),
        "gold_prob": float(np.exp(tok_lp.mean().item())),
        "pred_entropy": float(-(p * logp).sum()),
        "answer_margin": float(final[g0]) - float(tmp.max()),
        "is_top1": bool(top1 == g0),
        "n_tokens": int(ids_t.shape[1]),
        "rank_by_layer": np.asarray(rank_by_layer, dtype=np.int64),
    }

In [ ]:
# ── run both prompts for every fact ───────────────────────────────────────────────────
CACHE = OUT / f"prompt_sensitivity_step{STEP}.npz"
if CACHE.exists():
    _z = np.load(CACHE, allow_pickle=True)
    Z_SC, F_SC, Z_RBL, F_RBL = _z["zs"], _z["fs"], _z["zs_rbl"], _z["fs_rbl"]
    print(f"loaded cached results from {CACHE}")
else:
    zs, fs, zr, fr = [], [], [], []
    t0 = time.time()
    for i, r in rows.iterrows():
        a = lens(D.zero_shot_cloze(r), r["obj"])          # what the paper measures
        b = lens(fewshot.prompt(r), r["obj"])             # what the label was scored on
        for acc, rbl, res in ((zs, zr, a), (fs, fr, b)):
            acc.append({k: v for k, v in res.items() if k != "rank_by_layer"})
            rbl.append(res["rank_by_layer"])
        if i and i % 200 == 0:
            print(f"  {i}/{len(rows)}  ({time.time() - t0:.0f}s)")
    Z_SC = pd.DataFrame(zs).to_records(index=False)
    F_SC = pd.DataFrame(fs).to_records(index=False)
    Z_RBL, F_RBL = np.stack(zr), np.stack(fr)
    np.savez_compressed(CACHE, zs=Z_SC, fs=F_SC, zs_rbl=Z_RBL, fs_rbl=F_RBL,
                        ids=np.asarray(rows["id"]))
    print(f"done in {time.time() - t0:.0f}s -> {CACHE}")
ZS, FS = pd.DataFrame(Z_SC), pd.DataFrame(F_SC)
n_bound = Z_RBL.shape[1]
print(f"{len(ZS)} facts x {n_bound} boundaries, both prompts")
print(f"prompt length: zero-shot {ZS['n_tokens'].mean():.0f} tokens, "
      f"15-shot {FS['n_tokens'].mean():.0f} tokens")

In [ ]:
# ── Q1: per-fact agreement between the two prompts ────────────────────────────────────
lr_z = np.log10(ZS["gold_rank"].to_numpy() + 1.0)     # rank+1, matching paper_figures Fig. 1
lr_f = np.log10(FS["gold_rank"].to_numpy() + 1.0)
agree = {
    "gold_rank (log10, Spearman)": spearmanr(lr_z, lr_f)[0],
    "gold_prob (Spearman)":        spearmanr(ZS["gold_prob"], FS["gold_prob"])[0],
    "pred_entropy (Spearman)":     spearmanr(ZS["pred_entropy"], FS["pred_entropy"])[0],
    "answer_margin (Spearman)":    spearmanr(ZS["answer_margin"], FS["answer_margin"])[0],
}
print("rank correlation between the two prompts, per fact:")
for k, v in agree.items():
    print(f"  {k:32s} {v:+.3f}")
print(f"\nis_top1 agreement: {(ZS['is_top1'] == FS['is_top1']).mean():.1%}  "
      f"(zero-shot {ZS['is_top1'].mean():.1%} top-1, 15-shot {FS['is_top1'].mean():.1%})")
print(f"median gold rank:  zero-shot {ZS['gold_rank'].median():.0f}, "
      f"15-shot {FS['gold_rank'].median():.0f}")
print(f"median gold prob:  zero-shot {ZS['gold_prob'].median():.3f}, "
      f"15-shot {FS['gold_prob'].median():.3f}")

In [ ]:
# ── Q2: does Figure 1's depth profile survive the prompt change? ──────────────────────
# Same estimator as paper_figures.ipynb Fig. 1: log10(rank+1) and group medians.  Checkpoint
# balancing is neither needed nor possible here -- this is one checkpoint.
def profile(RBL):
    c = np.log10(RBL + 1.0)[correct_now]
    y = forget_next[correct_now]
    return np.median(c[~y], 0), np.median(c[y], 0)

zs_s, zs_f = profile(Z_RBL)
fs_s, fs_f = profile(F_RBL)
gap_z, gap_f = zs_f - zs_s, fs_f - fs_s

# One transition per fact at a single checkpoint, so a plain bootstrap over facts IS the
# fact-cluster bootstrap that Fig. 1 uses over the full panel.
rng = np.random.default_rng(0)

def boot_gap(RBL, n=400):
    c = np.log10(RBL + 1.0)[correct_now]
    y = forget_next[correct_now]
    out = np.full((n, c.shape[1]), np.nan)
    for b in range(n):
        s = rng.integers(0, len(c), len(c))
        cs, ys = c[s], y[s]
        if ys.any() and (~ys).any():
            out[b] = np.median(cs[ys], 0) - np.median(cs[~ys], 0)
    return np.nanpercentile(out, [2.5, 97.5], axis=0)

lo_z, hi_z = boot_gap(Z_RBL)
lo_f, hi_f = boot_gap(F_RBL)
r_curve = np.corrcoef(gap_z, gap_f)[0, 1]

xl = np.arange(n_bound)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15.5, 4.5),
                                    gridspec_kw={"width_ratios": [1, 1.15, 1.15]})

ax1.scatter(lr_z, lr_f, s=9, color=PURPLE, alpha=.35, edgecolor="none")
lim = [0, max(lr_z.max(), lr_f.max()) * 1.03]
ax1.plot(lim, lim, color=INK2, lw=1, ls=":")
ax1.set_xlim(lim)
ax1.set_ylim(lim)
ax1.set_xlabel(r"zero-shot  $\log_{10}(\mathrm{gold\ rank}+1)$")
ax1.set_ylabel(r"15-shot  $\log_{10}(\mathrm{gold\ rank}+1)$")
ax1.set_title(f"(a) Per fact: rho = {agree['gold_rank (log10, Spearman)']:.2f}",
              loc="left", fontsize=10)

ax2.plot(xl, zs_s, color=BLUE, lw=2, marker="o", ms=3.5, label="survives — zero-shot")
ax2.plot(xl, zs_f, color=ORANGE, lw=2, marker="o", ms=3.5, label="forgotten — zero-shot")
ax2.plot(xl, fs_s, color=BLUE, lw=2, ls="--", marker="s", ms=3.5, label="survives — 15-shot")
ax2.plot(xl, fs_f, color=ORANGE, lw=2, ls="--", marker="s", ms=3.5, label="forgotten — 15-shot")
ax2.axhline(0, color=INK2, lw=1, ls=":")
ax2.set_xlabel("layer boundary")
ax2.set_ylabel(r"median $\log_{10}(\mathrm{rank}+1)$")
ax2.set_title("(b) Depth profiles under both prompts", loc="left", fontsize=10)
ax2.legend(fontsize=7.5, loc="lower left")

ax3.axhline(0, color=INK2, lw=1)
ax3.fill_between(xl, lo_z, hi_z, color=BLUE, alpha=.15, linewidth=0)
ax3.fill_between(xl, lo_f, hi_f, color=ORANGE, alpha=.15, linewidth=0)
ax3.plot(xl, gap_z, color=BLUE, lw=2.2, marker="o", ms=4,
         label="zero-shot (the paper's Fig. 1)")
ax3.plot(xl, gap_f, color=ORANGE, lw=2.2, ls="--", marker="s", ms=4,
         label="15-shot (the scoring prompt)")
ax3.set_xlabel("layer boundary")
ax3.set_ylabel("gap  (forget − survive)")
ax3.set_title(f"(c) The claim: r = {r_curve:.2f} between the curves", loc="left", fontsize=10)
ax3.legend(fontsize=8, loc="upper left")

for ax in (ax2, ax3):
    ax.set_xticks(xl[::3])
for ax in (ax1, ax2, ax3):
    clean(ax)
fig.suptitle(f"Prompt sensitivity at step {STEP} — is the depth profile a property of the fact "
             "or of the prompt?", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

print("boundary   :", " ".join(f"{i:6d}" for i in xl))
print("gap 0-shot :", " ".join(f"{v:6.2f}" for v in gap_z))
print("gap 15-shot:", " ".join(f"{v:6.2f}" for v in gap_f))
_on_z = np.where(lo_z > 0)[0]
_on_f = np.where(lo_f > 0)[0]
print(f"\nonset (first boundary whose 95% CI clears 0): "
      f"zero-shot {_on_z[0] if len(_on_z) else None}, 15-shot {_on_f[0] if len(_on_f) else None}")
print(f"curve correlation across boundaries: r = {r_curve:.3f}")
print(f"mean |difference| in the gap:        {np.abs(gap_z - gap_f).mean():.3f} log-units")

## How to read the result

**Panel (c) decides it.** The paper's Figure 1 claims an *onset* at boundary 8 and a sustained
plateau after it. That is a claim about the fact, not about the prompt, only if the two curves
here have the same shape.

- **If `r` is high (≳0.9) and the onsets agree** — the caveat becomes a measured footnote:
  *"the depth profile is unchanged when the lens is run on the 15-shot scoring prompt
  (r = …, onset unchanged)"*. Nothing in `paper_figures.ipynb` needs to move.
- **If the curves diverge** — Figure 1 is partly a statement about the zero-shot prompt and
  must be scoped or recomputed. Note that switching to 15-shot is not automatically the right
  repair: under `drop-same-prop` the prefix is relation-dependent, so a 15-shot lens carries
  relation identity into the representation, and Fig. A3 shows relation identity is already the
  probe's dominant confound. Divergence is a reason to *scope the claim*, and only then to
  consider recomputing.

**Panel (a) is context, not the result.** Low per-fact agreement is expected and is not by
itself a problem — the few-shot block genuinely raises the gold answer, so ranks should shift.
The question is whether the *survive-vs-forget contrast* shifts with them.

### Scope of this check

- **One checkpoint.** If the curves agree here, the reasonable inference is that they agree
  throughout; it is not a proof. Re-run with a different `STEP` for a second point.
- **Answer position only.** The subject-position hidden states behind Fig. 3 and A3 are not
  covered. Locating the subject inside a 15-shot prompt needs a search restricted to the query
  suffix — the prefix length is known exactly, so it is mechanical, but it is not done here.
- **The neuron dump is not covered.** The same argument would need its own pass.
- This notebook only *measures* the sensitivity. It deliberately changes nothing in
  `paper_figures.ipynb`.